### P = NP?

Im Jahr 2000 veröffentlichte das Clay Mathematics Institute sieben Probleme der Mathematik, auf deren Lösung jeweils
ein Preisgeld von einer Million Dollar ausgesetzt ist (Millennium-Probleme). Eines davon ist die Frage **P = NP?**,
die bekannteste offene Frage der Informatik. Sie fragt, vereinfacht gesagt: Ist jedes Problem, dessen Lösung man
schnell **überprüfen** kann, auch schnell **lösbar**?

In diesem Notebook geht es um

- schnelle und langsame Algorithmen (polynomielle und exponentielle Laufzeit),
- die Klassen P und NP,
- NP-vollständige Probleme,
- Näherungsverfahren für Probleme, die man vermutlich nicht schnell exakt lösen kann.

#### Polynomiell und exponentiell

Ein Algorithmus hat **polynomielle Laufzeit**, wenn seine Laufzeit höchstens wie $n^k$ wächst, für eine feste Zahl
$k$ ($n$ ist die Größe der Eingabe). Beispiele sind $O(n)$, $O(n \log n)$, $O(n^2)$, $O(n^3)$. **Exponentielle
Laufzeit** wie $O(2^n)$ oder $O(n!)$ wächst so schnell, dass schon bei kleinen Eingaben kein Rechner mehr fertig
wird. Angenommen, ein Rechner schafft $10^9$ Schritte pro Sekunde:

In [1]:
import math

def dauer(schritte):
    sekunden = schritte / 1e9
    if sekunden < 60:
        return f'{sekunden:.2g} s'
    if sekunden < 3600 * 24 * 365:
        return f'{sekunden / 3600:.2g} h'
    return f'{sekunden / (3600 * 24 * 365):.2g} Jahre'

print(f'{"n":>4} {"n^2":>10} {"n^3":>10} {"2^n":>14} {"n!":>14}')
for n in [10, 20, 30, 50, 70]:
    print(f'{n:>4} {dauer(n ** 2):>10} {dauer(n ** 3):>10} {dauer(2 ** n):>14} {dauer(math.factorial(n)):>14}')

   n        n^2        n^3            2^n             n!
  10    1e-07 s    1e-06 s        1e-06 s       0.0036 s
  20    4e-07 s    8e-06 s        0.001 s       77 Jahre
  30    9e-07 s  2.7e-05 s          1.1 s  8.4e+15 Jahre
  50  2.5e-06 s  0.00013 s      3.1e+02 h  9.6e+47 Jahre
  70  4.9e-06 s  0.00034 s  3.7e+04 Jahre  3.8e+83 Jahre


Bei $n = 70$ braucht ein Algorithmus mit $n^3$ Schritten eine drittel Millisekunde, einer mit $2^n$ Schritten
37000 Jahre. Deshalb nennt man Probleme mit einem polynomiellen Algorithmus **effizient lösbar** oder salopp
„leicht“, Probleme, für die man nur exponentielle Algorithmen kennt, „schwer“.

### Entscheidungsprobleme

Wir betrachten nur **Entscheidungsprobleme**, also Probleme mit der Antwort Ja oder Nein (wie im Notebook zum
Halteproblem).

<img src="bild4.png" width="350">

- Gibt es in einem Graphen einen Weg von Knoten u nach Knoten v?
- Hat dieses Sudoku eine Lösung?
- Kann Weiß in dieser Schachstellung sicher gewinnen?
- Gibt es eine Rundreise durch diese Städte, die kürzer als 1000 km ist?

Viele Optimierungsprobleme lassen sich als Entscheidungsproblem formulieren. Statt „Wie kurz ist die kürzeste
Rundreise?“ fragt man „Gibt es eine Rundreise, die kürzer als $k$ ist?“. Kann man diese Frage schnell beantworten,
findet man die kürzeste Länge durch binäre Suche über $k$.

### Die Klassen P, NP, EXP und R

- **P** ist die Menge aller Entscheidungsprobleme, die sich in polynomieller Zeit lösen lassen. Beispiel: Gibt es
  einen Weg von u nach v? (Breitensuche)
- **EXP** ist die Menge aller Entscheidungsprobleme, die sich in exponentieller Zeit lösen lassen.
- **R** ist die Menge aller Entscheidungsprobleme, die sich überhaupt in endlicher Zeit lösen lassen. Das
  Halteproblem liegt nicht in R.
- **NP** liegt dazwischen. NP steht für *nichtdeterministisch polynomiell*.

#### NP: Raten und Prüfen

Man stellt sich einen Algorithmus vor, der unterwegs **raten** darf und dabei immer richtig rät (ein „lucky
algorithm“). Muss er sich z.B. bei einem Sudoku für eine Zahl in einem Feld entscheiden, nimmt er immer die, die zu
einer Lösung führt, falls es eine gibt. Unter allen Möglichkeiten findet er also garantiert einen Weg zum Ja.

<img src="bild2.png" width="300">

NP ist die Menge aller Entscheidungsprobleme, die ein solcher Algorithmus in polynomieller Zeit löst. Genauer darf
er bei jedem Schritt unter polynomiell vielen Möglichkeiten wählen.

Man kann sich das Raten auch ganz an den Anfang verlegt denken: Erst wird eine Lösung geraten, dann wird geprüft, ob
sie stimmt. Das führt zu einer gleichwertigen Definition, die man sich leichter merken kann:

> Ein Entscheidungsproblem ist in **NP**, wenn man für jede Ja-Antwort einen Lösungsvorschlag angeben kann, den man in
> polynomieller Zeit **überprüfen** kann.

**Sudoku:** Ein Sudoku zu lösen ist schwer (im Notebook zum Backtracking haben wir dafür alle Möglichkeiten
durchprobiert). Eine fertige Lösung zu prüfen ist leicht. Für ein $4 \times 4$-Sudoku:

In [2]:
def pruefe_sudoku(feld):
    n = len(feld)
    k = math.isqrt(n)
    soll = set(range(1, n + 1))
    zeilen = all(set(zeile) == soll for zeile in feld)
    spalten = all({feld[i][j] for i in range(n)} == soll for j in range(n))
    bloecke = all({feld[bi + i][bj + j] for i in range(k) for j in range(k)} == soll
                  for bi in range(0, n, k) for bj in range(0, n, k))
    return zeilen and spalten and bloecke

loesung = [[1, 2, 3, 4],
           [3, 4, 1, 2],
           [2, 1, 4, 3],
           [4, 3, 2, 1]]
falsch = [[1, 2, 3, 4],
          [2, 1, 4, 3],
          [3, 4, 1, 2],
          [4, 3, 2, 1]]
print(pruefe_sudoku(loesung), pruefe_sudoku(falsch))

True False


Die Prüfung schaut jede Zeile, Spalte und jeden Block einmal an, das geht für ein $n \times n$-Sudoku in $O(n^2)$.
Also ist Sudoku (verallgemeinert auf beliebig große Felder) in NP. Im zweiten Feld stimmen alle Zeilen und Spalten,
aber im Block links oben stehen die 1 und die 2 doppelt.

**Faktorisierung:** Hat die Zahl $n$ einen Teiler zwischen 2 und $k$? Einen Teiler zu finden ist schwer. Ist ein
Teiler vorgeschlagen, prüft eine einzige Division, ob er stimmt. Also ist das Problem in NP.

**Schach:** Ob Weiß in einer Stellung sicher gewinnt, lässt sich nicht einfach prüfen. Ein Gewinnzug allein reicht
als Nachweis nicht, man müsste alle Antworten von Schwarz mit allen weiteren Zügen durchgehen. Für Schach auf
$n \times n$-Brettern kennt man nur exponentielle Algorithmen, und man vermutet, dass es nicht in NP liegt. Bewiesen
ist das aber nicht.

Jedes Problem in P ist auch in NP: Man braucht nichts zu raten, man löst es einfach. Und jedes Problem in NP lässt
sich in exponentieller Zeit lösen, indem man alle Lösungsvorschläge durchprobiert. Also gilt
$P \subseteq NP \subseteq EXP$.

### Das SAT-Problem

Gegeben ist ein logischer Ausdruck mit Variablen $x_1, x_2, \dots$, die True oder False sein können. Gibt es eine
Belegung der Variablen, bei der der Ausdruck True ist? Das ist das **Erfüllbarkeitsproblem** (satisfiability, SAT).
Meist schreibt man den Ausdruck als UND-Verknüpfung von **Klauseln**, jede Klausel ist eine ODER-Verknüpfung von
**Literalen**. Ein Literal ist eine Variable oder ihre Verneinung ($\overline{x}$ heißt nicht $x$).

<img src="bild5.png" width="550">

Im Code schreiben wir eine Klausel als Liste von Zahlen: $3$ steht für $x_3$, $-1$ für $\overline{x_1}$.

In [3]:
formel = [[-1, 2, 3], [2, -3, 4], [1, -2]]

def wert(formel, belegung):
    '''belegung[i] ist der Wert von x_i, Index 0 wird nicht benutzt'''
    return all(any(belegung[abs(l)] if l > 0 else not belegung[abs(l)] for l in klausel) for klausel in formel)

print(wert(formel, [None, True, True, False, False]))
print(wert(formel, [None, False, True, False, False]))

True
False


Eine Belegung zu **prüfen** geht schnell: jede Klausel einmal ansehen. SAT ist also in NP. Eine erfüllende Belegung
zu **finden** ist schwer. Der naheliegende Algorithmus probiert alle $2^n$ Belegungen der $n$ Variablen durch:

In [4]:
import itertools

def sat_brute_force(formel, n):
    for werte in itertools.product([False, True], repeat=n):
        belegung = [None] + list(werte)
        if wert(formel, belegung):
            return belegung
    return None

print(sat_brute_force(formel, 4))

[None, False, False, False, False]


Für Formeln, die nicht erfüllbar sind, muss er wirklich alle $2^n$ Belegungen prüfen. Wir messen das mit zufälligen
Formeln, die je drei Literale pro Klausel und siebenmal so viele Klauseln wie Variablen haben (solche Formeln sind
fast nie erfüllbar):

In [5]:
import random
import time

def zufallsformel(n, anzahl_klauseln, seed):
    rnd = random.Random(seed)
    return [[v * rnd.choice([1, -1]) for v in rnd.sample(range(1, n + 1), 3)] for k in range(anzahl_klauseln)]

for n in [10, 12, 14, 16]:
    f = zufallsformel(n, 7 * n, seed=n)
    start = time.time()
    ergebnis = sat_brute_force(f, n)
    print(n, 'erfüllbar' if ergebnis else 'nicht erfüllbar', f'{time.time() - start:.3f} s')

10 nicht erfüllbar 0.005 s
12 nicht erfüllbar 0.026 s
14 nicht erfüllbar 0.103 s


16 nicht erfüllbar 0.385 s


Zwei Variablen mehr bedeuten viermal so viele Belegungen, die Zeit wächst entsprechend. Mit 50 Variablen wären es
$2^{50} \approx 10^{15}$ Belegungen. Es gibt viel geschicktere SAT-Löser, die in der Praxis oft Formeln mit Millionen
Variablen lösen. Ein Algorithmus, der **jede** Formel in polynomieller Zeit entscheidet, ist aber nicht bekannt.

### Stand heute

<img src="bild7.png" width="350">

Man weiß, dass $P \subseteq NP \subseteq EXP$ gilt und dass $P \ne EXP$ ist: Es gibt Probleme, die nachweislich nicht
in polynomieller Zeit lösbar sind. Man weiß aber nicht, ob $P = NP$ ist und ob $NP = EXP$ ist. Die meisten Forscher
vermuten $P \ne NP$.

### NP-vollständige Probleme

**Stephen Cook** (und unabhängig Leonid Levin) bewies 1971: SAT ist **NP-vollständig**. Das heißt

1. SAT ist in NP,
2. jedes andere Problem in NP lässt sich auf SAT **reduzieren** (SAT ist **NP-schwer**).

**Reduzieren** heißt: Es gibt einen Algorithmus mit polynomieller Laufzeit, der jede Frage des einen Problems in eine
Frage des anderen Problems mit derselben Antwort übersetzt. Beispiel: das **Clique-Problem** (Gibt es in einem
Graphen $k$ Knoten, die alle miteinander verbunden sind?).

<img src="bild9.png" width="700">

Könnte man SAT schnell lösen, dann auch das Clique-Problem: erst übersetzen, dann SAT lösen.

#### Ein Beispiel für eine Reduktion: Graphen färben

Kann man die Knoten eines Graphen mit $k$ Farben so färben, dass benachbarte Knoten verschiedene Farben haben? Wir
übersetzen diese Frage in eine SAT-Formel. Die Variable $x_{v,c}$ bedeutet: Knoten $v$ hat Farbe $c$. Die Klauseln:

- Jeder Knoten hat mindestens eine Farbe: $(x_{v,1} \lor x_{v,2} \lor x_{v,3})$.
- Kein Knoten hat zwei Farben: $(\overline{x_{v,1}} \lor \overline{x_{v,2}})$ usw.
- Benachbarte Knoten haben nicht dieselbe Farbe: $(\overline{x_{u,c}} \lor \overline{x_{v,c}})$ für jede Kante
  $(u, v)$ und jede Farbe $c$.

In [6]:
def faerbung_nach_sat(knoten, kanten, k):
    nummer = {}
    for v in knoten:
        for c in range(k):
            nummer[(v, c)] = len(nummer) + 1
    formel = []
    for v in knoten:
        formel.append([nummer[(v, c)] for c in range(k)])
        for c1 in range(k):
            for c2 in range(c1 + 1, k):
                formel.append([-nummer[(v, c1)], -nummer[(v, c2)]])
    for u, v in kanten:
        for c in range(k):
            formel.append([-nummer[(u, c)], -nummer[(v, c)]])
    return formel, nummer

knoten = ['A', 'B', 'C', 'D']
kanten = [('A', 'B'), ('B', 'C'), ('C', 'D'), ('D', 'A'), ('A', 'C')]
for k in [2, 3]:
    formel, nummer = faerbung_nach_sat(knoten, kanten, k)
    belegung = sat_brute_force(formel, len(nummer))
    print(k, 'Farben:', len(nummer), 'Variablen,', len(formel), 'Klauseln')
    if belegung is None:
        print('   nicht färbbar')
    else:
        print('   ', {v: c for (v, c), i in nummer.items() if belegung[i]})

2 Farben: 8 Variablen, 18 Klauseln
   nicht färbbar
3 Farben: 12 Variablen, 31 Klauseln
    {'A': 2, 'B': 1, 'C': 0, 'D': 1}


Mit 2 Farben ist die Formel nicht erfüllbar: A, B, C bilden ein Dreieck. Mit 3 Farben liefert die erfüllende
Belegung direkt eine Färbung. Die Übersetzung selbst ist schnell, sie erzeugt nur polynomiell viele Klauseln.

Es gibt auch eine Reduktion von SAT auf Clique und auf das Färben mit 3 Farben, also sind diese Probleme ebenfalls
NP-vollständig. Inzwischen kennt man tausende NP-vollständige Probleme, z.B. das Rucksackproblem, das Problem des
Handlungsreisenden (als Entscheidungsproblem), verallgemeinertes Sudoku und Dominating Set. Alle hängen zusammen:
**Findet jemand für eines davon einen polynomiellen Algorithmus, dann gibt es einen für alle Probleme in NP, und es
gilt P = NP.**

<img src="bild8.png" width="500">

Ein Problem heißt **NP-schwer**, wenn sich jedes Problem aus NP darauf reduzieren lässt, auch wenn es selbst nicht in
NP liegt. NP-vollständig sind die NP-schweren Probleme, die in NP liegen.

#### Suchen ohne zu suchen?

Man kann P = NP? auch so formulieren: Kann man jedes Suchproblem lösen, ohne wirklich zu suchen? Kann man die Nadel
im Heuhaufen finden, ohne den Heuhaufen zu durchwühlen, vielleicht mit einem Magneten?

Bei einem Problem kennt man so einen Magneten: Ob eine Zahl eine Primzahl ist, kann man herausfinden, ohne nach
Teilern zu suchen. Nach dem **kleinen Satz von Fermat** gilt für jede Primzahl $p$ und jedes $a$ mit $1 \le a < p$:
$a^{p-1} \;\%\; p = 1$. Ist das Ergebnis nicht 1, kann $p$ keine Primzahl sein, auch wenn wir keinen Teiler kennen.
`pow(a, e, m)` berechnet $a^e \;\%\; m$ auch für große Zahlen schnell.

In [7]:
print(pow(2, 90, 91))
print(pow(2, 96, 97))
print(pow(2, 560, 561), 561 == 3 * 11 * 17)

64
1
1 True


$2^{90} \;\%\; 91 = 64$, also ist 91 keine Primzahl (tatsächlich $91 = 7 \cdot 13$). Bei 97 kommt 1 heraus, 97 ist
eine Primzahl. Der Test allein reicht aber nicht: Bei 561 kommt ebenfalls 1 heraus, obwohl $561 = 3 \cdot 11 \cdot 17$
ist. Solche Zahlen heißen Carmichael-Zahlen. 2002 fanden Manindra Agrawal, Neeraj Kayal und Nitin Saxena einen
Primzahltest, der auf einer Verallgemeinerung des Satzes von Fermat beruht und für jede Zahl in polynomieller Zeit
die richtige Antwort liefert (AKS-Test). Damit ist bewiesen: Das Primzahlproblem ist in P.

Für das Faktorisieren, also das Finden der Teiler, kennt man dagegen keinen polynomiellen Algorithmus. Darauf beruht
die Sicherheit von RSA.

### Näherungsverfahren

Für NP-vollständige Probleme kennt man nur exponentielle Algorithmen. In der Praxis muss man sie trotzdem lösen,
z.B. Stundenpläne, Touren für Lieferwagen oder die Verteilung von Funkfrequenzen. Man begnügt sich dann oft mit
einer **Näherungslösung**, die schnell berechnet wird und meist gut, aber nicht immer optimal ist. Eine einfache
Strategie ist **Greedy** (gierig): in jedem Schritt das wählen, was im Moment am besten aussieht, und die Wahl nie
zurücknehmen.

#### Greedy-Färbung: das Vier-Farben-Problem

Eine Landkarte soll so gefärbt werden, dass benachbarte Länder verschiedene Farben haben. Nach dem
**Vier-Farben-Satz** (1976 mit Computerhilfe bewiesen) reichen dafür immer vier Farben. Ob drei Farben reichen, zu
entscheiden, ist dagegen NP-vollständig. Wir färben die Karte der Bundesländer als Graph: Jedes Land ist ein Knoten,
benachbarte Länder sind verbunden.

In [8]:
nachbarn = {
    'SH': ['HH', 'NI', 'MV'], 'HH': ['SH', 'NI'], 'MV': ['SH', 'NI', 'BB'],
    'NI': ['SH', 'HH', 'MV', 'BB', 'ST', 'TH', 'HE', 'NW', 'HB'], 'HB': ['NI'],
    'BB': ['MV', 'NI', 'ST', 'SN', 'BE'], 'BE': ['BB'], 'ST': ['NI', 'BB', 'SN', 'TH'],
    'SN': ['BB', 'ST', 'TH', 'BY'], 'TH': ['NI', 'ST', 'SN', 'BY', 'HE'],
    'HE': ['NI', 'TH', 'BY', 'BW', 'RP', 'NW'], 'NW': ['NI', 'HE', 'RP'],
    'RP': ['NW', 'HE', 'BW', 'SL'], 'SL': ['RP'], 'BW': ['RP', 'HE', 'BY'], 'BY': ['BW', 'HE', 'TH', 'SN']}

def greedy_faerbung(G, reihenfolge):
    farbe = {}
    for v in reihenfolge:
        belegt = {farbe[w] for w in G[v] if w in farbe}
        c = 0
        while c in belegt:
            c += 1
        farbe[v] = c
    return farbe

f = greedy_faerbung(nachbarn, sorted(nachbarn))
print(max(f.values()) + 1, 'Farben:', f)

4 Farben: {'BB': 0, 'BE': 1, 'BW': 0, 'BY': 1, 'HB': 0, 'HE': 2, 'HH': 0, 'MV': 1, 'NI': 3, 'NW': 0, 'RP': 1, 'SH': 2, 'SL': 0, 'SN': 2, 'ST': 1, 'TH': 0}


Greedy geht die Länder in einer festen Reihenfolge durch und gibt jedem Land die kleinste Farbnummer, die keiner
seiner schon gefärbten Nachbarn hat. Das geht in polynomieller Zeit. In alphabetischer Reihenfolge braucht es 4
Farben. Geht es mit 3? Das prüfen wir mit der Reduktion auf SAT:

In [9]:
kanten = [(a, b) for a in nachbarn for b in nachbarn[a] if a < b]
formel, nummer = faerbung_nach_sat(list(nachbarn), kanten, 3)
print(len(nummer), 'Variablen', len(formel), 'Klauseln')

48 Variablen 151 Klauseln


Für 48 Variablen müsste unser Brute-Force-Löser bis zu $2^{48}$ Belegungen prüfen, das dauert viel zu lange. Für 16
Länder und 3 Farben gibt es aber nur $3^{16} \approx 43$ Millionen Färbungen. Wir probieren sie geschickter mit
Backtracking durch: Länder der Reihe nach färben und abbrechen, sobald zwei Nachbarn gleich gefärbt sind.

In [10]:
def faerbe(G, k, reihenfolge, farbe=None):
    if farbe is None:
        farbe = {}
    if len(farbe) == len(reihenfolge):
        return dict(farbe)
    v = reihenfolge[len(farbe)]
    for c in range(k):
        if all(farbe.get(w) != c for w in G[v]):
            farbe[v] = c
            ergebnis = faerbe(G, k, reihenfolge, farbe)
            if ergebnis:
                return ergebnis
            del farbe[v]
    return None

print('3 Farben:', faerbe(nachbarn, 3, sorted(nachbarn)))
print('4 Farben:', faerbe(nachbarn, 4, sorted(nachbarn)) is not None)

3 Farben: None
4 Farben: True


Mit 3 Farben geht es nicht, die alphabetische Greedy-Färbung ist hier also optimal. Das hängt aber von der
Reihenfolge ab:

In [11]:
import collections

zufall = random.Random(0)
anzahl = collections.Counter()
for versuch in range(2000):
    reihenfolge = list(nachbarn)
    zufall.shuffle(reihenfolge)
    anzahl[max(greedy_faerbung(nachbarn, reihenfolge).values()) + 1] += 1
print(sorted(anzahl.items()))

[(4, 1319), (5, 677), (6, 4)]


Bei 2000 zufälligen Reihenfolgen braucht Greedy meist 4 Farben, in etwa einem Drittel der Fälle aber 5 und manchmal
sogar 6. Eine bewährte Regel: die Knoten mit den meisten Nachbarn zuerst färben.

#### Greedy für Dominating Set

Eine **dominierende Menge** (Dominating Set) ist eine Menge von Knoten, so dass jeder Knoten des Graphen in der Menge
liegt oder einen Nachbarn in der Menge hat. Anwendung: Wo stellt man möglichst wenige Feuerwachen auf, so dass jeder
Ort eine Wache hat oder an einen Ort mit Wache grenzt? Eine möglichst kleine dominierende Menge zu finden, ist
NP-schwer.

Greedy: Wähle immer den Knoten, der die meisten noch nicht abgedeckten Knoten abdeckt (sich selbst und seine
Nachbarn).

In [12]:
def greedy_dominating(G):
    offen = set(G)
    auswahl = []
    while offen:
        v = max(sorted(G), key=lambda v: len(({v} | set(G[v])) & offen))
        auswahl.append(v)
        offen -= {v} | set(G[v])
    return auswahl

def optimal_dominating(G):
    for k in range(1, len(G) + 1):
        for menge in itertools.combinations(sorted(G), k):
            abgedeckt = set(menge)
            for v in menge:
                abgedeckt |= set(G[v])
            if abgedeckt == set(G):
                return list(menge)

spinne = {'A': ['B', 'C', 'D'], 'B': ['A', 'E'], 'C': ['A', 'F'], 'D': ['A', 'G'],
          'E': ['B'], 'F': ['C'], 'G': ['D']}
print('Greedy:', greedy_dominating(spinne), ' optimal:', optimal_dominating(spinne))
print('Bundesländer, Greedy:', greedy_dominating(nachbarn), ' optimal:', optimal_dominating(nachbarn))

Greedy: ['A', 'B', 'C', 'D']  optimal: ['B', 'C', 'D']
Bundesländer, Greedy: ['NI', 'BW', 'BB', 'RP']  optimal: ['BB', 'BW', 'NI', 'RP']


Im Graphen „Spinne“ (A in der Mitte mit drei Beinen B-E, C-F, D-G) wählt Greedy zuerst A, weil A die meisten Knoten
abdeckt. Danach sind E, F und G noch offen, und dafür braucht es drei weitere Knoten. Optimal sind B, C, D: Sie
decken A, ihre Beine und sich selbst ab. Greedy ist schnell, liefert aber nicht immer die kleinste Lösung. Die
optimale Lösung haben wir durch Ausprobieren aller Teilmengen gefunden, das ist exponentiell.

Für die Bundesländer findet Greedy vier Länder: Jedes Bundesland ist Niedersachsen, Baden-Württemberg, Brandenburg
oder Rheinland-Pfalz oder grenzt an eines davon. Hier ist Greedy optimal, mit drei Ländern geht es nicht.

### Zusammenfassung

- Polynomielle Laufzeit gilt als effizient, exponentielle Laufzeit ist schon bei kleinen Eingaben nicht mehr
  praktikabel.
- **P**: Entscheidungsprobleme, die in polynomieller Zeit lösbar sind. **NP**: Entscheidungsprobleme, bei denen man
  eine Lösung in polynomieller Zeit überprüfen kann. Es gilt $P \subseteq NP \subseteq EXP$, ob $P = NP$ ist, weiß
  niemand.
- **SAT** ist NP-vollständig (Cook 1971): Es liegt in NP, und jedes Problem in NP lässt sich darauf reduzieren.
  Tausende weitere Probleme sind NP-vollständig, z.B. Clique, Färben mit 3 Farben, Rucksack, Dominating Set. Ein
  polynomieller Algorithmus für eines davon würde P = NP beweisen.
- Für NP-vollständige Probleme benutzt man in der Praxis oft **Näherungsverfahren** wie **Greedy**. Sie sind schnell,
  liefern aber nicht immer die beste Lösung.